#**Imports**

In [ ]:
# Typical Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
# Modeling & preprocessing import
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

#**Define Custom Functions**

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def regression_metrics(y_true, y_pred, label='', verbose = True, output_dict=False):
  # Get metrics
  mae = mean_absolute_error(y_true, y_pred)
  mse = mean_squared_error(y_true, y_pred)
  rmse = mean_squared_error(y_true, y_pred)
  r_squared = r2_score(y_true, y_pred)
  if verbose == True:
    # Print Result with Label and Header
    header = "-"*60
    print(header, f"Regression Metrics: {label}", header, sep='\n')
    print(f"- MAE = {mae:,.3f}")
    print(f"- MSE = {mse:,.3f}")
    print(f"- RMSE = {rmse:,.3f}")
    print(f"- R^2 = {r_squared:,.3f}")
  if output_dict == True:
      metrics = {'Label':label, 'MAE':mae,
                 'MSE':mse, 'RMSE':rmse, 'R^2':r_squared}
      return metrics
def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose = True,
                        output_frame=False):
  # Get predictions for training data
  y_train_pred = reg.predict(X_train)
  # Call the helper function to obtain regression metrics for training data
  results_train = regression_metrics(y_train, y_train_pred, verbose = verbose,
                                     output_dict=output_frame,
                                     label='Training Data')
  print()
  # Get predictions for test data
  y_test_pred = reg.predict(X_test)
  # Call the helper function to obtain regression metrics for test data
  results_test = regression_metrics(y_test, y_test_pred, verbose = verbose,
                                  output_dict=output_frame,
                                    label='Test Data' )
  # Store results in a dataframe if ouput_frame is True
  if output_frame:
    results_df = pd.DataFrame([results_train,results_test])
    # Set the label as the index
    results_df = results_df.set_index('Label')
    # Set index.name to none to get a cleaner looking result
    results_df.index.name=None
    # Return the dataframe
    return results_df.round(3)

#**Mount Drive and Load Data**

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Using the recommended file structure.
fpath = "/content/drive/MyDrive/AXSOSACADEMY/02-MachineLearning/Week06/auto_mpg_dirty.csv"
df = pd.read_csv(fpath)
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        355 non-null    float64
 5   acceleration  375 non-null    float64
 6   model year    398 non-null    int64  
 7   origin        333 non-null    object 
 8   car name      398 non-null    object 
dtypes: float64(5), int64(2), object(2)
memory usage: 28.1+ KB


,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130.0,3504.0,12.0,70,America,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,America,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,America,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,America,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,America,ford torino


##  Model Pipelines

### شو الجديد؟
بدل ما نعمل preprocessing منفصل عن النموذج —
بنجمعهم في pipeline وحدة.

Pipeline structure:

pipe[0]  → preprocessor (feature names هون)

pipe[-1] → model (coefficients/importances هون)

In [ ]:
# Arrange Data into Features Matrix and Target Vector
y = df['mpg']
X = df.drop(columns = ['mpg','car name'])
# Split the data for validation
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [ ]:
# Saving list of categorical columns
cat_cols = X_train.select_dtypes('object').columns
# Constructing categorical preprocessing objects
cat_imputer = SimpleImputer(strategy='constant', fill_value='MISSING')
ohe_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_pipe = make_pipeline(cat_imputer,ohe_encoder)
cat_tuple = ('cat',cat_pipe, cat_cols)
cat_tuple

('cat',
 Pipeline(steps=[('simpleimputer',
                  SimpleImputer(fill_value='MISSING', strategy='constant')),
                 ('onehotencoder',
                  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
 Index(['origin'], dtype='object'))

In [ ]:
# Save list of numeric columns
num_cols = X_train.select_dtypes('number').columns
# Constructing numeric preprocesssing objects
num_imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
num_pipe = make_pipeline(num_imputer, scaler)
num_tuple = ('num',num_pipe, num_cols)
num_tuple

('num',
 Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                 ('standardscaler', StandardScaler())]),
 Index(['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration',
        'model year'],
       dtype='object'))

In [ ]:
# Define a column transformer
preprocessor  = ColumnTransformer([num_tuple, cat_tuple],
                                  verbose_feature_names_out=False)

In [ ]:
# Linear Regression Pipeline
# Make & Fit pipeline
linreg_pipe = make_pipeline(preprocessor, LinearRegression())
linreg_pipe.fit(X_train, y_train)
evaluate_regression(linreg_pipe, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 2.730
- MSE = 13.019
- RMSE = 13.019
- R^2 = 0.791

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.731
- MSE = 12.247
- RMSE = 12.247
- R^2 = 0.784


In [ ]:
#  جيب الـ feature names والـ coefficients من الـ pipeline
# Feature names from preprocessor (index 0)
feature_names = linreg_pipe[0].get_feature_names_out()

# Coefficients from model (index -1)
coeffs = pd.Series(linreg_pipe[-1].coef_, index=feature_names)
coeffs

,0
cylinders,-0.294326
displacement,-0.763556
horsepower,-1.383290
weight,-3.070077
acceleration,-0.545605
model year,2.715286
origin_America,-1.481413
origin_Asia,0.987628
origin_Europe,0.782469
origin_MISSING,-0.288684


In [ ]:
#Decision Tree Pipeline
tree_pipe = make_pipeline(preprocessor, DecisionTreeRegressor(random_state=42))
tree_pipe.fit(X_train, y_train)
evaluate_regression(tree_pipe, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 0.000
- MSE = 0.000
- RMSE = 0.000
- R^2 = 1.000

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.463
- MSE = 12.522
- RMSE = 12.522
- R^2 = 0.779


### Key Takeaway

When using a pipeline:

pipe[0]  → always the preprocessor → use for feature names

pipe[-1] → always the model → use for .coef_ or .feature_importances_

This approach keeps our code clean and prevents
data leakage between train and test sets ✅